# TC Tracking Runner - E3SM Hindcast (ne30pg2)

This notebook only configures and calls `scripts/run_process_tc_track_e3sm.py` to process TC tracks with TempestExtremes.

**Pipeline:** `DetectNodes` -> `StitchNodes` -> `HistogramNodes`
**Output layout:** `/global/cfs/cdirs/e3sm/S2S2D/post_process/CASE/MEMBER/post/atm/tc-analysis/`

Run this notebook first to generate the stitched track text files and histogram NetCDF files used by the diagnostics notebook.


In [1]:
import os
import sys
import subprocess
from pathlib import Path

# Resolve native-library data from the interpreter running this kernel and set
# the environment variables BEFORE importing Cartopy/GDAL/PyProj or any package
# that transitively imports them.  Jupyter can inherit a stale CONDA_PREFIX from
# its server environment, which causes those libraries to load stale native
# data unless these variables are set first.
_env_prefix = sys.prefix
_proj_path = os.path.join(_env_prefix, "share", "proj")
_gdal_path = os.path.join(_env_prefix, "share", "gdal")
if not os.path.isfile(os.path.join(_proj_path, "proj.db")):
    raise FileNotFoundError(f"PROJ data not found under {_env_prefix}")
if not os.path.isfile(os.path.join(_gdal_path, "header.dxf")):
    raise FileNotFoundError(f"GDAL data not found under {_env_prefix}")
os.environ["CONDA_PREFIX"] = _env_prefix
os.environ["PROJ_LIB"] = _proj_path
os.environ["PROJ_DATA"] = _proj_path
os.environ["GDAL_DATA"] = _gdal_path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

# Python interpreter for the tracking script.
# When the kernel IS e3sm_analysis, sys.executable is already correct.
PYTHON = sys.executable
print(f"PYTHON      : {PYTHON}")
print(f"CONDA_PREFIX: {os.environ.get('CONDA_PREFIX', '(not set — fallback paths used)')}")

from esp_lab import data_access_e3sm as data_access


PYTHON      : /global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin/python
CONDA_PREFIX: /global/homes/z/zhan391/.conda/envs/e3sm_analysis


## Configuration — fill in all fields before running

Fields left as `None` will cause the **Validate** cell to raise an error.

### Warm-core method sets

`PARSETS_TO_RUN` controls which warm-core definitions are evaluated by the Dry-run and Production cells below. The preview cells in Sections 1–3 use the first entry in the list.

Examples:

```python
PARSETS_TO_RUN = ["set3"]                         # run one method
PARSETS_TO_RUN = ["set3", "set5"]                 # primary + sensitivity
PARSETS_TO_RUN = ["set1", "set2", "set3",
                  "set4", "set5"]                 # compare all methods
```

Each entry is passed to the tracking script through `--parset`.

### Guidance from Zarzycki and Ullrich (2017,doi:10.1002/2016GL071606)

Vertically integrated geopotential-thickness warm-core metrics generally performed better than single-level temperature metrics.

The tested warm-core variables ranked approximately as:

`DZ300500 > T400 > DZ200500 > DZ400500 > T500 > T300`

`DZ300500` was the preferred warm-core diagnostic across the tested reanalyses, while `T400` was the best single-level temperature diagnostic.

The optimized JRA/DZ300500 configuration used `wcFOmag = -6 m`.

Warm-core strength and contour-distance thresholds are among the most sensitive tracker parameters, so they should be validated for each model, resolution, and dataset.

### Interpretation of the available sets

- **set3:** `Z300 - Z500`, threshold `-6.0`  
  Primary/reference method. This most closely matches the preferred and optimized `DZ300500` configuration in the paper.

- **set5:** `T400`, threshold `-0.4`  
  Recommended independent sensitivity method. `T400` was the best single-level temperature option in the paper.

- **set1:** `Z200 - Z500`, threshold `-8.0`  
  Geopotential-thickness alternative. This class of diagnostic was tested, but it was less strongly supported than `Z300 - Z500`.

- **set2:** `T200 - T500`, threshold `-0.8`  
  Experimental two-level temperature-difference method. This exact diagnostic was not evaluated in the paper.

- **set4:** `T300 - T500`, threshold `-0.6`  
  Experimental two-level temperature-difference method. This exact diagnostic was also not evaluated in the paper.

### Recommended production usage

```python
PARSETS_TO_RUN = ["set3", "set5"]
```

Use `set3` as the primary result and `set5` as a robustness check. Run all five sets when explicitly evaluating sensitivity to the warm-core definition.

The remaining detection and stitching thresholds are Yeager-style and apply consistently to all sets.

### Unit requirement

The current `Z200`, `Z300`, and `Z500` fields are geopotential heights in meters.

If the fields instead contain geopotential in `m² s⁻²`, convert the warm-core threshold accordingly:

`-6.0 m ≈ -58.8 m² s⁻²`

### Evaluation guidance

Raw TC count is more sensitive to tracker settings than integrated metrics such as TC days and ACE. Method comparisons should therefore examine storm count together with TC days, ACE, track lifetime, latitude distribution, and false-alarm characteristics.

In [2]:
# ------------------------------------------------------------------ #
#  PATHS  (all required)
# ------------------------------------------------------------------ #
OUTDIR       = Path("/global/cfs/cdirs/e3sm/S2S2D/post_process")
CONNECT_FILE = Path("/global/cfs/cdirs/e3sm/zhan391/TempestExtremes/grid_info/outCS_ne30pg2_connect.txt")
SCRIPT       = Path("/global/homes/z/zhan391/code/ESP-Lab/scripts/run_process_tc_track_e3sm.py")

# ------------------------------------------------------------------ #
#  CASES - built from multi-case prefixes + init tags
# ------------------------------------------------------------------ #
# Multi-case E3SM hindcasts. Add more entries here as new simulations
# land under their sim_dir with the same directory convention.
E3SM_CASES = {
    "E3SM-FOSIRL": {
        "sim_dir": Path("/global/cfs/cdirs/e3smdata/simulations/S2S2D"),
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL",
        "cache_tag": "JRA55_FOSIRL",
        "display_name": "E3SMv3-FOSIRL",
        "grid": "ne30pg2",
        "stream_tag": "eam.h2",
        "phis_stream_tag": "eam.h0",
    },
    "E3SM-Reanalysis": {
        "sim_dir": Path("/global/cfs/cdirs/e3sm/S2S2D/simulation"),
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce",
        "cache_tag": "Reanalysis",
        "display_name": "E3SMv3-Reanalysis",
        "grid": "ne30pg2",
        "stream_tag": "eam.h3",
        "phis_stream_tag": "eam.h0",
    },
#    "E3SM-4DEnVarOcn": {
#        "sim_dir": Path("/global/cfs/cdirs/e3sm/S2S2D/simulation"),
#        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_4DEnVarOcn",
#        "cache_tag": "4DEnVarOcn",
#        "display_name": "E3SMv3-4DEnVarOcn",
#        "grid": "ne30pg2",
#        "stream_tag": "eam.h2",
#        "phis_stream_tag": "eam.h0",
#    },
}

years  = 1980
yeare  = 2018
yexcl  = None          # set to a year int to exclude, or None

init_months = [5, 11]  # May and November starts

lead_years = [y for y in np.arange(years, yeare + 1) if y != yexcl]

CASES_BY_E3SM_CASE = {}
CASE_TO_E3SM_CASE = {}
CASE_SIM_DIRS = {}
CASE_GRIDS = {}
CASE_STREAM_TAGS = {}
CASE_PHIS_STREAM_TAGS = {}
for case_key, case_info in E3SM_CASES.items():
    case_prefix = case_info["case_prefix"]
    sim_dir = Path(case_info["sim_dir"])
    grid = case_info["grid"]
    stream_tag = case_info["stream_tag"]
    phis_stream_tag = case_info["phis_stream_tag"]
    case_names = []
    for init_month in init_months:
        init_tags = data_access.build_init_tags(lead_years, init_month)
        case_names += [f"{case_prefix}_{tag}" for tag in init_tags]
    CASES_BY_E3SM_CASE[case_key] = case_names
    for case_name in case_names:
        CASE_TO_E3SM_CASE[case_name] = case_key
        CASE_SIM_DIRS[case_name] = sim_dir
        CASE_GRIDS[case_name] = grid
        CASE_STREAM_TAGS[case_name] = stream_tag
        CASE_PHIS_STREAM_TAGS[case_name] = phis_stream_tag

CASES = [case for case_names in CASES_BY_E3SM_CASE.values() for case in case_names]

print(f"E3SM case groups: {list(CASES_BY_E3SM_CASE)}")
for case_key, case_names in CASES_BY_E3SM_CASE.items():
    case_info = E3SM_CASES[case_key]
    print(f"  {case_key}: {len(case_names)} cases")
    print(f"    SIM_DIR   : {case_info['sim_dir']}")
    print(f"    First few : {case_names[:3]}")
    print(f"    Last few  : {case_names[-3:]}")
print(f"Total cases across groups: {len(CASES)}")

# ------------------------------------------------------------------ #
#  ENSEMBLE  (required)
# ------------------------------------------------------------------ #

case_nens = 10
members   = [f"EN{i:02d}" for i in range(case_nens)]
MEMBERS   = members
NENS      = None   # e.g. 3 for a quick test

# ------------------------------------------------------------------ #
#  PARSET
# ------------------------------------------------------------------ #

# List of PARSET (warm-core method set) values to loop over in the
# Dry-run/Production cells below (and used by the preview cells in
# Sections 1-3, which check the first entry).  Set to a single-item list
# (e.g. ["set2"]) to run one set, or e.g.
# ["set1", "set2", "set3", "set4", "set5"] to run every warm-core method set
# in one notebook execution (e.g. for the method-comparison figure).  Each
# entry is passed downstream to the tracking script via --parset.
# The thresholds below are Yeager-style and apply to all sets.
# Current Z200/Z300/Z500 fields are in m; if using geopotential in m2/s2,
# convert the Z warm-core threshold from -6.0 m to -58.8 m2/s2.
#PARSETS_TO_RUN  = ["set1", "set2", "set3", "set4", "set5"]
PARSETS_TO_RUN  = ["set3"]

# Optional per-parset overrides for the warm-core definition, passed downstream
# to the tracking script as --wc1/--wc2/--wc-mag/--wc-vc.  Leave a parset out
# (or a key unset) to use the script's built-in PARSETS default for that field.
# Example:
#   WC_OVERRIDES = {"set2": {"wc_mag": -0.5}}
WC_OVERRIDES = {
    "set1": {"wc_mag": -16.0},
    "set2": {"wc_mag": -1.0},
    "set3": {"wc_mag": -6.0},
    "set4": {"wc_mag": -0.6},
    "set5": {"wc_mag": -0.4}
}


# Explicit warm-core field definitions per parset: (wc1, wc2), with wc2=None
# for single-field sets (e.g. set5).  This is the single source of truth for
# which fields each parset needs - it drives both the variable-availability
# check in Section 2 below (via TC_VARS_BY_PARSET) AND the --wc1/--wc2 flags
# passed downstream to the tracking script (see build_cmd in Section 4), so
# what gets validated always matches what TempestExtremes actually receives.
WC_FIELDS_BY_PARSET = {
    "set1": ("Z200", "Z500"),
    "set2": ("T200", "T500"),
    "set3": ("Z300", "Z500"),
    "set4": ("T300", "T500"),
    "set5": ("T400", None),
}

# Required eam.h2 variables per parset, derived from WC_FIELDS_BY_PARSET plus
# the fields common to every set (PSL, UBOT, VBOT).
TC_VARS_BY_PARSET = {
    parset: {"PSL", "UBOT", "VBOT"} | {v for v in fields if v is not None}
    for parset, fields in WC_FIELDS_BY_PARSET.items()
}

WORKERS         = 10

# ------------------------------------------------------------------ #
#  TRACKING THRESHOLDS
#  Yeager-style stitching settings used for every warm-core method set.
#  MIN_WIND is a tracking-continuity threshold, not a Saffir-Simpson
#  category threshold; category coloring should be handled downstream.
# ------------------------------------------------------------------ #

PSL_FO_MAG      = 200.0
PSL_FO_DIST     = 8.0
WC_FO_DIST      = 8.0
WC_MAX_OFFSET   = 3.0
MERGE_DIST      = 6.0
TRAJ_RANGE      = 10.0
TRAJ_MIN_LENGTH = "10"
TRAJ_MAX_GAP    = "3"
MAX_TOPO        = 150.0
MAX_LAT         = 50.0
MIN_WIND        = 8.0
SCI_DIST        = 9

# ------------------------------------------------------------------ #
#  TEMPESTEXTREMES + NCO bin directory
#  Auto-resolved from CONDA_PREFIX when running under e3sm_analysis kernel.
#  Hardcoded fallback for any other kernel.
# ------------------------------------------------------------------ #

REQUIRED_COMMANDS = [
    "DetectNodes",
    "StitchNodes",
    "HistogramNodes",
    "GenerateCSMesh",
    "GenerateVolumetricMesh",
    "GenerateConnectivityFile",
    "ncks",
    "ncwa",
]

_ENV_BIN = os.path.join(os.environ.get("CONDA_PREFIX", ""), "bin")
if not all(os.path.exists(os.path.join(_ENV_BIN, cmd)) for cmd in REQUIRED_COMMANDS):
    _ENV_BIN = "/global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin"

TE_BIN  = _ENV_BIN   # DetectNodes / StitchNodes / HistogramNodes
NCO_BIN = _ENV_BIN   # ncks / ncwa

print(f"_ENV_BIN : {_ENV_BIN}")


E3SM case groups: ['E3SM-FOSIRL', 'E3SM-Reanalysis']
  E3SM-FOSIRL: 78 cases
    SIM_DIR   : /global/cfs/cdirs/e3smdata/simulations/S2S2D
    First few : ['WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100', 'WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1981050100', 'WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1982050100']
    Last few  : ['WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_2016110100', 'WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_2017110100', 'WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_2018110100']
  E3SM-Reanalysis: 78 cases
    SIM_DIR   : /global/cfs/cdirs/e3sm/S2S2D/simulation
    First few : ['WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce_1980050100', 'WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce_1981050100', 'WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce_1982050100']
    Last few  : ['WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce_2016110100', 'WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce_2017110100', 'WCYCL20TR_ne30p

## Validate — run this before anything else

In [3]:
# ---- enforce all required fields are set ----
_required = {
    "CASES":           CASES,
    "CASE_SIM_DIRS":   CASE_SIM_DIRS,
    "CASE_GRIDS":      CASE_GRIDS,
    "CASE_STREAM_TAGS": CASE_STREAM_TAGS,
    "CASE_PHIS_STREAM_TAGS": CASE_PHIS_STREAM_TAGS,
    "OUTDIR":          OUTDIR,
    "CONNECT_FILE":    CONNECT_FILE,
    "SCRIPT":          SCRIPT,
    "MEMBERS":         MEMBERS,
    "TE_BIN":          TE_BIN,
    "PARSETS_TO_RUN":  PARSETS_TO_RUN,
    "WORKERS":         WORKERS,
    "PSL_FO_MAG":      PSL_FO_MAG,
    "PSL_FO_DIST":     PSL_FO_DIST,
    "WC_FO_DIST":      WC_FO_DIST,
    "WC_MAX_OFFSET":   WC_MAX_OFFSET,
    "MERGE_DIST":      MERGE_DIST,
    "TRAJ_RANGE":      TRAJ_RANGE,
    "TRAJ_MIN_LENGTH": TRAJ_MIN_LENGTH,
    "TRAJ_MAX_GAP":    TRAJ_MAX_GAP,
    "MAX_TOPO":        MAX_TOPO,
    "MAX_LAT":         MAX_LAT,
    "MIN_WIND":        MIN_WIND,
    "SCI_DIST":        SCI_DIST,
}
_missing = [k for k, v in _required.items() if v is None]
if _missing:
    raise ValueError(
        "The following required fields are still None - fill them in the Configuration cell:\n"
        + "\n".join(f"  {k}" for k in _missing)
    )

# ---- check each case directory exists under its case-specific sim_dir ----
_errors = []
for _case_key, _case_names in CASES_BY_E3SM_CASE.items():
    _sim_dir = Path(E3SM_CASES[_case_key]["sim_dir"])
    if not _sim_dir.is_dir():
        _errors.append(f"SIM_DIR does not exist for {_case_key}: {_sim_dir}")
        continue
    for _case in _case_names:
        _cdir = _sim_dir / _case
        if not _cdir.is_dir():
            _errors.append(f"Case directory not found for {_case_key}: {_cdir}")
if not SCRIPT.is_file():
    _errors.append(f"Tracking script not found: {SCRIPT}")
if _errors:
    raise FileNotFoundError("Path validation failed:\n" + "\n".join(_errors))

# ---- check PARSETS_TO_RUN / WC_OVERRIDES ----
_valid_parsets = {"set1", "set2", "set3", "set4", "set5"}
_invalid_parsets = [p for p in PARSETS_TO_RUN if p not in _valid_parsets]
if _invalid_parsets:
    raise ValueError(
        f"Invalid PARSETS_TO_RUN values: {_invalid_parsets}; "
        f"must be one of {sorted(_valid_parsets)}"
    )
_invalid_override_keys = {
    k: sorted(set(v) - {"wc1", "wc2", "wc_mag", "vc"})
    for k, v in WC_OVERRIDES.items()
}
_invalid_override_keys = {k: v for k, v in _invalid_override_keys.items() if v}
if _invalid_override_keys:
    raise ValueError(
        f"WC_OVERRIDES has unsupported keys: {_invalid_override_keys}; "
        "supported keys are wc1, wc2, wc_mag, vc"
    )
_invalid_field_parsets = [p for p in WC_FIELDS_BY_PARSET if p not in _valid_parsets]
if _invalid_field_parsets:
    raise ValueError(
        f"Invalid WC_FIELDS_BY_PARSET keys: {_invalid_field_parsets}; "
        f"must be one of {sorted(_valid_parsets)}"
    )
_missing_field_parsets = [p for p in PARSETS_TO_RUN if p not in WC_FIELDS_BY_PARSET]
if _missing_field_parsets:
    raise ValueError(
        f"PARSETS_TO_RUN references parsets missing from WC_FIELDS_BY_PARSET: "
        f"{_missing_field_parsets}"
    )

# ---- check required executables individually ----
_missing_commands = [
    cmd for cmd in REQUIRED_COMMANDS
    if not os.path.exists(os.path.join(TE_BIN, cmd)) and not os.path.exists(os.path.join(NCO_BIN, cmd))
]
if _missing_commands:
    raise FileNotFoundError(
        f"Required executables not found in TE_BIN/NCO_BIN: {_missing_commands}"
    )

# ---- resolve member list from the first case ----
_ref_case = CASES[0]
_ref_case_dir = CASE_SIM_DIRS[_ref_case] / _ref_case
_all_members = sorted(p.name for p in _ref_case_dir.iterdir()
                      if p.is_dir() and p.name.startswith("EN"))
members_avail = list(MEMBERS) if MEMBERS else _all_members
if not members_avail:
    raise ValueError(f"No EN* member directories found in {_ref_case_dir}")
if NENS is not None:
    members_avail = members_avail[:NENS]

# ---- validate every (case, member) combination has a history directory ----
_missing_history = []
for _case in CASES:
    _sim_dir = CASE_SIM_DIRS[_case]
    for _member in members_avail:
        _hist_dir = _sim_dir / _case / _member / "archive" / "atm" / "hist"
        if not _hist_dir.is_dir():
            _missing_history.append(str(_hist_dir))
if _missing_history:
    print(f"WARNING: {len(_missing_history)} (case, member) history directories are missing:")
    for _h in _missing_history[:20]:
        print(f"  {_h}")
    if len(_missing_history) > 20:
        print(f"  ... and {len(_missing_history) - 20} more")

print("Configuration valid")
print(f"  Case groups   : {list(CASES_BY_E3SM_CASE)}")
for _case_key, _case_names in CASES_BY_E3SM_CASE.items():
    _case_info = E3SM_CASES[_case_key]
    _display = _case_info.get("display_name", _case_key)
    _sim_dir = Path(_case_info["sim_dir"])
    print(f"    {_case_key} ({_display}): {len(_case_names)} total  ({_case_names[0]}  ...  {_case_names[-1]})")
    print(f"      SIM_DIR: {_sim_dir}")
    print(f"      Grid   : {_case_info['grid']}")
    print(f"      Streams: {_case_info['stream_tag']} / {_case_info['phis_stream_tag']}")
print(f"  Cases total   : {len(CASES)}")
print(f"  OUTDIR        : {OUTDIR}")
print(f"  CONNECT_FILE  : {CONNECT_FILE}  {'exists' if CONNECT_FILE.exists() else '(will auto-generate)'}")
print(f"  SCRIPT        : {SCRIPT}")
print(f"  Members       : {members_avail}  (NENS={NENS})")
print(f"  Parsets to run: {PARSETS_TO_RUN}")
print(f"  WC fields     : {WC_FIELDS_BY_PARSET}")
if WC_OVERRIDES:
    print(f"  WC overrides  : {WC_OVERRIDES}")
print(f"  Workers       : {WORKERS}")
print("  Tracking args : "
      f"psl={PSL_FO_MAG}/{PSL_FO_DIST}, "
      f"wc_dist={WC_FO_DIST}, wc_offset={WC_MAX_OFFSET}, "
      f"range={TRAJ_RANGE}, minlen={TRAJ_MIN_LENGTH}, maxgap={TRAJ_MAX_GAP}, "
      f"wind>={MIN_WIND}, sci_dist={SCI_DIST}, maxlat={MAX_LAT}, maxtopo={MAX_TOPO}")
print(f"\n  Example input : {CASE_SIM_DIRS[CASES[0]] / CASES[0] / members_avail[0] / 'archive' / 'atm' / 'hist'}")
print(f"  Example output: {OUTDIR / CASES[0] / members_avail[0] / 'post' / 'atm' / 'tc-analysis'}")


Configuration valid
  Case groups   : ['E3SM-FOSIRL', 'E3SM-Reanalysis']
    E3SM-FOSIRL (E3SMv3-FOSIRL): 78 total  (WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100  ...  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_2018110100)
      SIM_DIR: /global/cfs/cdirs/e3smdata/simulations/S2S2D
      Grid   : ne30pg2
      Streams: eam.h2 / eam.h0
    E3SM-Reanalysis (E3SMv3-Reanalysis): 78 total  (WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce_1980050100  ...  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce_2018110100)
      SIM_DIR: /global/cfs/cdirs/e3sm/S2S2D/simulation
      Grid   : ne30pg2
      Streams: eam.h3 / eam.h0
  Cases total   : 156
  OUTDIR        : /global/cfs/cdirs/e3sm/S2S2D/post_process
  CONNECT_FILE  : /global/cfs/cdirs/e3sm/zhan391/TempestExtremes/grid_info/outCS_ne30pg2_connect.txt  exists
  SCRIPT        : /global/homes/z/zhan391/code/ESP-Lab/scripts/run_process_tc_track_e3sm.py
  Members       : ['EN00', 'EN01', 'EN02', 'EN03', 'EN04', 'EN05', 'EN0

## 1.  Survey available `eam.h2` files

In [4]:
# Survey h2/h0 file counts for first two cases (all members)
for case in CASES[:2]:
    case_dir = CASE_SIM_DIRS[case] / case
    stream_tag = CASE_STREAM_TAGS[case]
    phis_stream_tag = CASE_PHIS_STREAM_TAGS[case]
    print(f"\nCase : {case}")
    print(f"SIM_DIR: {CASE_SIM_DIRS[case]}")
    for member in members_avail:
        hist_dir = case_dir / member / "archive" / "atm" / "hist"
        h2_files = sorted(hist_dir.glob(f"*.{stream_tag}.*.nc"))
        h0_files = sorted(hist_dir.glob(f"*.{phis_stream_tag}.*.nc"))
        print(f"  {member}")
        print(f"    {stream_tag}  : {len(h2_files)} files", end="")
        if h2_files:
            print(f"  [{h2_files[0].name}  ...  {h2_files[-1].name}]")
        else:
            print("  <none found>")
        print(f"    {phis_stream_tag} : {len(h0_files)} files", end="")
        if h0_files:
            print(f"  [{h0_files[0].name}  ...  {h0_files[-1].name}]")
        else:
            print("  <none found>")



Case : WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100
SIM_DIR: /global/cfs/cdirs/e3smdata/simulations/S2S2D
  EN00
    eam.h2  : 25 files  [WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN00.eam.h2.1980-05-01-00000.nc  ...  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN00.eam.h2.1982-04-21-00000.nc]
    eam.h0 : 24 files  [WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN00.eam.h0.1980-05.nc  ...  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN00.eam.h0.1982-04.nc]
  EN01
    eam.h2  : 25 files  [WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN01.eam.h2.1980-05-01-00000.nc  ...  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN01.eam.h2.1982-04-21-00000.nc]
    eam.h0 : 24 files  [WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN01.eam.h0.1980-05.nc  ...  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN01.eam.h0.1982-04.nc]
  EN02
    eam.h2  : 25 files

## 2.  Verify TC variables in `eam.h2`

Required variables are checked for every set in `PARSETS_TO_RUN` (`set3` uses `PSL`, `UBOT`, `VBOT`, `Z300`, `Z500`).  
`PHIS` comes from a static file extracted from `eam.h0`.

In [5]:
case = CASES[0]
case_dir = CASE_SIM_DIRS[case] / case
stream_tag = CASE_STREAM_TAGS[case]
phis_stream_tag = CASE_PHIS_STREAM_TAGS[case]
member   = members_avail[0]
hist_dir = case_dir / member / "archive" / "atm" / "hist"

h2_files = sorted(hist_dir.glob(f"*.{stream_tag}.*.nc"))
assert h2_files, f"No {stream_tag} files in {hist_dir}"
with xr.open_dataset(h2_files[0], decode_times=False) as ds_h2:
    all_vars = set(ds_h2.data_vars) | set(ds_h2.coords)
    sizes = dict(ds_h2.sizes)

print(f"Case   : {case}")
print(f"SIM_DIR: {CASE_SIM_DIRS[case]}")
print(f"Member : {member}")
print(f"Stream : {stream_tag}  ({h2_files[0].name})")
print(f"  dims   : {sizes}")
for _parset in PARSETS_TO_RUN:
    _required_vars = TC_VARS_BY_PARSET[_parset]
    _found = _required_vars & all_vars
    _missing = _required_vars - all_vars
    print(f"  [{_parset}] found  : {sorted(_found)}")
    print(f"  [{_parset}] missing: {sorted(_missing) or 'none'}")

h0_files = sorted(hist_dir.glob(f"*.{phis_stream_tag}.*.nc"))
assert h0_files, f"No {phis_stream_tag} files in {hist_dir}"
with xr.open_dataset(h0_files[0], decode_times=False) as ds_h0:
    phis_ok = "PHIS" in ds_h0.data_vars or "PHIS" in ds_h0.coords
print(f"\nStream : {phis_stream_tag}  ({h0_files[0].name})")
print(f"  PHIS  : {'present' if phis_ok else 'MISSING  <-- problem'}")


Case   : WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100
SIM_DIR: /global/cfs/cdirs/e3smdata/simulations/S2S2D
Member : EN00
Stream : eam.h2  (WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN00.eam.h2.1980-05-01-00000.nc)
  dims   : {'ncol': 21600, 'lev': 80, 'ilev': 81, 'cosp_prs': 7, 'nbnd': 2, 'cosp_tau': 7, 'cosp_ht': 40, 'cosp_temp': 40, 'cosp_sr': 15, 'cosp_htmisr': 16, 'cosp_tau_modis': 7, 'cosp_reffice': 6, 'cosp_reffliq': 6, 'cosp_iwp_modis': 7, 'cosp_lwp_modis': 7, 'time': 120, 'P3_input_dim': 16, 'P3_output_dim': 32, 'cosp_scol': 10, 'cosp_sza': 5}
  [set3] found  : ['PSL', 'UBOT', 'VBOT', 'Z300', 'Z500']
  [set3] missing: none

Stream : eam.h0  (WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN00.eam.h0.1980-05.nc)
  PHIS  : present


## 3.  Check connectivity file

TempestExtremes requires a connectivity file for the native ne30pg2 unstructured grid.  
If it is missing, the tracking script will auto-generate it via `--grid`.

In [6]:
if CONNECT_FILE.exists():
    size_mb = CONNECT_FILE.stat().st_size / 1e6
    print(f"Connect file : {CONNECT_FILE}")
    print(f"  Size : {size_mb:.1f} MB  ✓")
    with open(CONNECT_FILE) as f:
        for i, line in enumerate(f):
            print(f"  line {i+1}: {line.rstrip()}")
            if i >= 3:
                break
else:
    print(f"Connect file not found: {CONNECT_FILE}")
    _grids = sorted(set(CASE_GRIDS.values()))
    print(f"It will be auto-generated via --grid {_grids[0] if len(_grids) == 1 else _grids} when the tracking script runs.")

Connect file : /global/cfs/cdirs/e3sm/zhan391/TempestExtremes/grid_info/outCS_ne30pg2_connect.txt
  Size : 1.9 MB  ✓
  line 1: #TempestGridConnectivityFileV2.0
  line 2: 1,21600
  line 3: 3.15740716043964e+02,-3.49114391634599e+01,5.18997015195786e-04,4,2,3,10918,17883
  line 4: 3.17240673319773e+02,-3.55819651497593e+01,5.27971397106035e-04,4,1,4,5,17884


## 4.  Dry-run — preview commands without executing

In [7]:
import time

def build_cmd(
    case: str,
    members: list,
    parset: str,
    *,
    sim_dir: Path,
    grid: str,
    stream_tag: str,
    phis_stream_tag: str,
    dry_run: bool,
    force: bool = False,
) -> list:
    """Build CLI command for a single case + member list + warm-core parset."""
    cmd = [
        PYTHON, str(SCRIPT),
        "--sim-dir",         str(sim_dir),
        "--outdir",          str(OUTDIR),
        "--cases",           case,
        "--members",         *members,
        "--parset",          parset,
        "--stream-tag",      stream_tag,
        "--phis-stream-tag", phis_stream_tag,
        "--connect-file",    str(CONNECT_FILE),
        "--grid",            grid,
        "--te-bin",          TE_BIN,
        "--workers",         str(WORKERS),
        "--psl-fo-mag",     str(PSL_FO_MAG),
        "--psl-fo-dist",    str(PSL_FO_DIST),
        "--wc-fo-dist",     str(WC_FO_DIST),
        "--wc-max-offset",  str(WC_MAX_OFFSET),
        "--merge-dist",     str(MERGE_DIST),
        "--traj-range",     str(TRAJ_RANGE),
        "--traj-min-length", str(TRAJ_MIN_LENGTH),
        "--traj-max-gap",   str(TRAJ_MAX_GAP),
        "--max-topo",       str(MAX_TOPO),
        "--max-lat",        str(MAX_LAT),
        "--min-wind",       str(MIN_WIND),
        "--sci-dist",       str(SCI_DIST),
        "--verbose",
    ]
    if NCO_BIN:
        cmd += ["--nco-bin", NCO_BIN]
    if NENS is not None:
        cmd += ["--nens", str(NENS)]

    # Explicit warm-core field/threshold setup, passed downstream to the
    # script.  wc1/wc2 always come from WC_FIELDS_BY_PARSET (the same
    # definitions validated against eam.h2 in Section 2), so what gets
    # checked there always matches what's requested from TempestExtremes.
    # wc_mag/vc may additionally be overridden per parset via WC_OVERRIDES.
    overrides = dict(WC_OVERRIDES.get(parset, {}))
    _wc1_default, _wc2_default = WC_FIELDS_BY_PARSET.get(parset, (None, None))
    overrides.setdefault("wc1", _wc1_default)
    overrides.setdefault("wc2", _wc2_default)

    if overrides.get("wc1") is not None:
        cmd += ["--wc1", str(overrides["wc1"])]
    if "wc2" in overrides:
        cmd += ["--wc2", "" if overrides["wc2"] is None else str(overrides["wc2"])]
    if overrides.get("wc_mag") is not None:
        cmd += ["--wc-mag", str(overrides["wc_mag"])]
    if "vc" in overrides:
        cmd += ["--wc-vc", "" if overrides["vc"] is None else str(overrides["vc"])]

    if dry_run:
        cmd += ["--dry-run"]
    if force:
        cmd += ["--force"]
    return cmd


def run_cases(
    cases_to_run,
    members_to_run,
    parset: str,
    *,
    sim_dir: Path,
    grid: str,
    stream_tag: str,
    phis_stream_tag: str,
    dry_run: bool,
    force: bool = False,
    label: str = "",
):
    """Loop over cases, run the tracking script for one parset, print per-case progress."""
    tag = "DRY-RUN" if dry_run else "RUN"
    label_text = f" {label}" if label else ""
    print(f"{tag}{label_text} [{parset}]: {len(cases_to_run)} cases x {len(members_to_run)} members")
    print(f"  SIM_DIR: {sim_dir}" + (f"  |  force={force}" if not dry_run else ""))
    print(f"  Grid/streams: {grid}, {stream_tag} / {phis_stream_tag}")
    print()

    results_log = {}
    t0_total = time.time()

    for i, case in enumerate(cases_to_run, 1):
        t0 = time.time()
        print(f"[{i:3d}/{len(cases_to_run)}]  {case}")

        cmd = build_cmd(
            case,
            members_to_run,
            parset,
            sim_dir=sim_dir,
            grid=grid,
            stream_tag=stream_tag,
            phis_stream_tag=phis_stream_tag,
            dry_run=dry_run,
            force=force,
        )
        result = subprocess.run(cmd, capture_output=True, text=True)

        elapsed = time.time() - t0
        status  = "OK" if result.returncode == 0 else "FAILED"
        results_log[case] = result.returncode

        for line in result.stdout.splitlines():
            if any(kw in line for kw in ("Summary", "OK", "Skipped", "Failed", "No data", "Dry-run", "Would process")):
                print(f"         {line.strip()}")

        print(f"         -> {status}  ({elapsed:.0f}s)\n")

        if result.returncode != 0:
            print("  === STDERR ===")
            print(result.stderr[-1000:])

    elapsed_total = time.time() - t0_total
    n_ok     = sum(1 for rc in results_log.values() if rc == 0)
    n_failed = sum(1 for rc in results_log.values() if rc != 0)
    print(f"\n{'='*60}")
    print(f"{tag}{label_text} [{parset}]  {len(cases_to_run)} cases  |  OK {n_ok}  |  failed {n_failed}  |  {elapsed_total/60:.1f} min")
    if n_failed:
        print("Failed cases:")
        for case, rc in results_log.items():
            if rc != 0:
                print(f"  {case}  (exit {rc})")
    return results_log


# ---- Dry-run: same case/member/parset selection as the real run ----
# Adjust these to test a subset before committing to the full run:
#   cases_to_run   = CASES_BY_E3SM_CASE[case_key][:3]
#   members_to_run = ["EN00", "EN01"]
members_to_run = members_avail

dry_run_log = {}
for case_key, cases_to_run in CASES_BY_E3SM_CASE.items():
    case_info = E3SM_CASES[case_key]
    case_label = case_info.get("display_name", case_key)
    sim_dir = Path(case_info["sim_dir"])
    grid = case_info["grid"]
    stream_tag = case_info["stream_tag"]
    phis_stream_tag = case_info["phis_stream_tag"]
    dry_run_log[case_key] = {}
    print(f"\n=== E3SM CASE: {case_key} ({case_label}) ===")
    for _parset in PARSETS_TO_RUN:
        print(f"\n--- PARSET: {_parset} ---")
        dry_run_log[case_key][_parset] = run_cases(
            cases_to_run,
            members_to_run,
            _parset,
            sim_dir=sim_dir,
            grid=grid,
            stream_tag=stream_tag,
            phis_stream_tag=phis_stream_tag,
            dry_run=True,
            label=case_key,
        )



=== E3SM CASE: E3SM-FOSIRL (E3SMv3-FOSIRL) ===

--- PARSET: set3 ---
DRY-RUN E3SM-FOSIRL [set3]: 78 cases x 10 members
  SIM_DIR: /global/cfs/cdirs/e3smdata/simulations/S2S2D
  Grid/streams: ne30pg2, eam.h2 / eam.h0

[  1/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100
         -> OK  (1s)

[  2/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1981050100
         -> OK  (0s)

[  3/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1982050100
         -> OK  (0s)

[  4/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1983050100
         -> OK  (0s)

[  5/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1984050100
         -> OK  (1s)

[  6/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1985050100
         -> OK  (0s)

[  7/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1986050100
         -> OK  (0s)

[  8/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1987050100
         -> OK  (0s)

[  9/78]  WCYCL20TR_ne30pg2_r05_IcoswI

## 5. Run TC Tracking

> **Prerequisites:**
> - `DetectNodes`, `StitchNodes`, `HistogramNodes` must be on `PATH` or available through `TE_BIN`.
> - Each member can take several minutes; use a batch job for large ensembles.
> - To re-run existing output, change `FORCE = False` to `FORCE = True`.


In [ ]:
# ---- Production run ----
# Adjust to process a subset:
#   cases_to_run   = CASES_BY_E3SM_CASE[case_key][:3]
#   members_to_run = ["EN00", "EN01"]
members_to_run = members_avail

FORCE = False   # set True only when deliberately overwriting existing output
DRY_RUN = False  # set False only when ready for the real production run

run_log = {}
for case_key, cases_to_run in CASES_BY_E3SM_CASE.items():
    case_info = E3SM_CASES[case_key]
    case_label = case_info.get("display_name", case_key)
    sim_dir = Path(case_info["sim_dir"])
    grid = case_info["grid"]
    stream_tag = case_info["stream_tag"]
    phis_stream_tag = case_info["phis_stream_tag"]
    run_log[case_key] = {}
    print(f"\n=== E3SM CASE: {case_key} ({case_label}) ===")
    for _parset in PARSETS_TO_RUN:
        print(f"\n--- PARSET: {_parset} ---")
        run_log[case_key][_parset] = run_cases(
            cases_to_run,
            members_to_run,
            _parset,
            sim_dir=sim_dir,
            grid=grid,
            stream_tag=stream_tag,
            phis_stream_tag=phis_stream_tag,
            dry_run=DRY_RUN,
            force=FORCE,
            label=case_key,
        )



=== E3SM CASE: E3SM-FOSIRL (E3SMv3-FOSIRL) ===

--- PARSET: set3 ---
RUN E3SM-FOSIRL [set3]: 78 cases x 10 members
  SIM_DIR: /global/cfs/cdirs/e3smdata/simulations/S2S2D  |  force=False
  Grid/streams: ne30pg2, eam.h2 / eam.h0

[  1/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100
         -> OK  (1s)

[  2/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1981050100
         -> OK  (1s)

[  3/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1982050100
         -> OK  (1s)

[  4/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1983050100
         -> OK  (0s)

[  5/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1984050100
         -> OK  (0s)

[  6/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1985050100
         -> OK  (1s)

[  7/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1986050100
         -> OK  (1s)

[  8/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1987050100
         -> OK  (1s)

[  9/78]  WCYCL20TR_ne30pg